In [1]:
import os
os.chdir('D:\\ai-engineering-buildcamp-codespace\\03-agents\\03-agents-framework\\')
print(os.getcwd())

D:\ai-engineering-buildcamp-codespace\03-agents\03-agents-framework


In [2]:
import pandas as pd
import requests
from pathlib import Path
from urllib.parse import urlparse
from markitdown import MarkItDown
from typing import Any, Dict, Iterable, List
import json
from pydantic import BaseModel, Field
from typing import Literal
from minsearch import AppendableIndex
from gitsource import GithubRepositoryDataReader, chunk_documents

In [3]:
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

True

In [4]:
from openai import OpenAI
openai_client  = OpenAI()

In [5]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex


reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [6]:
query = 'how do I use evidently to monitor my machine learning models?'

In [7]:
index.search(query)

[{'start': 0,
  'content': 'You can score your text by downloading and using ML models from HuggingFace. This lets you apply any criteria from the source model, e.g. classify texts by emotion. There are:\n\n* Ready-to-use descriptors that wrap a specific model,\n\n* A general interface to call other suitable models you select.\n\n**Pre-requisites**:\n\n* You know how to use [descriptors](/docs/library/descriptors) to evaluate text data.\n\n## Imports\n\n```python\nfrom evidently.descriptors import HuggingFace, HuggingFaceToxicity\n```\n\n<Accordion title="Toy data to run the example" defaultOpen={false}>\n  To generate toy data and create a Dataset object:\n\n  ```python\n  import pandas as pd\n\n  from evidently import Dataset\n  from evidently import DataDefinition\n\n  data = [\n      ["Why is the sky blue?", \n       "The sky is blue because molecules in the air scatter blue light from the sun more than they scatter red light.", \n       "because air scatters blue light more"],\n  

In the above setup the search results give you alot of contents and now we try to trimp those contents and highlight the most important information

In [11]:
search_results = index.search(query)

In [ ]:
from minsearch import Highlighter, Tokenizer
# we can customize the stop words by creating a new set that combines the default stop words with our custom ones
# we remove the stopwords from the content field to improve the relevance of the search results
from minsearch.tokenizer import DEFAULT_ENGLISH_STOP_WORDS

In [9]:
# we remove standard stop words and also the word 'evidently' to improve the relevance of the search results, 
# since it is likely to appear in many documents and not provide useful information for distinguishing between them
stopwords = DEFAULT_ENGLISH_STOP_WORDS | {'evidently'}

# we create a highlighter that will highlight the relevant parts of the content field in the search results,
#  using a tokenizer that removes stop words and applies stemming to improve the relevance of the highlighted snippets
#snippet_size is the number of characters to include in the highlighted snippet,
#  and max_matches is the maximum number of matches to highlight in each document
highlighter = Highlighter(
    highlight_fields=['content'],
    max_matches=3,
    snippet_size=50,
    tokenizer=Tokenizer(stemmer='snowball', stop_words=stopwords)
)

In [12]:
highlighter.highlight(query, search_results)

[{'start': 0,
  'content': {'matches': ['...downloading and using ML **models** from HuggingFace. This l...',
    '...criteria from the source **model**, e.g. classify texts by ...',
    '.... There are:\n\n* Ready-to-**use** descriptors that wrap a ...'],
   'total_matches': 11},
  'title': 'Use HuggingFace models',
  'description': 'How to use models from HuggingFace as evaluators.',
  'filename': 'metrics/customize_hf_descriptor.mdx'},
 {'start': 1500,
  'content': {'matches': ['...t-in evaluators for some **models**. You can call them like ...',
    '...\n</Info>\n\nAlternatively, **use** the general `HuggingFace...',
    '...to call a specific named **model**. The **model** you **use** must ...'],
   'total_matches': 10},
  'title': 'Use HuggingFace models',
  'description': 'How to use models from HuggingFace as evaluators.',
  'filename': 'metrics/customize_hf_descriptor.mdx'},
 {'start': 6000,
  'content': {'matches': ['...1. </li><li>[HuggingFace **Model**](https://huggingface

In [13]:
# Purpose: Retrieve the complete, original file content by filename
# What it contains:
# Complete, unchunked documents
# Key: filename → Value: full content
# The file_index is not redundant - it serves a different purpose than the search index. Think of it as:

# Search index = Library catalog (finds what you need)
# File index = Book shelf (retrieves the complete book)
file_index = {doc['filename']: doc['content'] for doc in parsed_docs}

In [14]:
file_index['docs/setup/self-hosting.mdx']

'<Info>\n  This page explains how to self-host the lightweight open-source platform. [Contact us](https://www.evidentlyai.com/get-demo) for Enterprise version with extra features and support. Compare [OSS vs. Enterprise/Cloud](/faq/oss_vs_cloud).\n</Info>\n\nIn addition to using Evidently Python library, you can self-host the UI Service to get a monitoring Dashboard and organize the results of your evaluations. This is optional: you can also view evaluation results in Python or export to JSON or HTML.\n\nTo get a self-hostable Dashboard, you must:\n\n- Create a Workspace (local or remote) to store your data.\n- Launch the UI service.\n\n## 1. Create a Workspace\n\n<Tip>\n  Sign up for a free [Evidently Cloud](cloud) account to get a managed version instantly.\n</Tip>\n\nOnce you [install Evidently](/docs/setup/installation), you will need to create a `workspace`. This designates a remote or local directory where you will store the evaluation results (JSON Reports called `snapshots`), t

In [15]:
from typing import Any, Dict, List


class SearchTools:
    """
    Provides search and file retrieval utilities over an indexed data store.
    """

    def __init__(
        self,
        index: Any,
        highlighter: Any,
        file_index: Dict[str, str]
    ) -> None:
        """
        Initialize the SearchTools instance.

        Args:
            index: Searchable index providing a `search` method.
            highlighter: Highlighter used to annotate search results.
            file_index (Dict[str, str]): Mapping of filenames to file contents.
        """
        self.index = index
        self.highlighter = highlighter
        self.file_index = file_index

    def search(self, query: str) -> List[Dict[str, Any]]:
        """
        Search the index for results matching a query and highlight them.

        Args:
            query (str): The search query to look up in the index.

        Returns:
            List[Dict[str, Any]]: A list of highlighted search result objects.
        """
        search_results = self.index.search(
            query=query,
            num_results=5
        )

        return self.highlighter.highlight(query, search_results)

    def get_file(self, filename: str) -> str:
        """
        Retrieve a file's contents by filename.

        Args:
            filename (str): The filename of the file to retrieve.

        Returns:
            str: The file contents if found, otherwise an error message.
        """
        if filename in self.file_index:
            return self.file_index[filename]
        return f"file {filename} does not exist"

In [16]:
search_tools = SearchTools(index, highlighter, file_index)

here we are modifying our instructions 

After we retreive relatevent chunk information along with file location, we will ask LLM to get the documents through get_file tool

In [18]:
instructions = """
You're a documentation assistant.

Answer the user question using only the documentation knowledge base.

Make 3 iterations:

1) First iteration:
   - Perform one search using the search tool to identify potentially relevant documents.
   - Explain (in 2–3 sentences) why this search query is appropriate for the user question.

2) Second iteration:
   - Analyze the results from the previous search.
   - Based on the filenames or documents returned, perform:
       - Up to 2 additional search queries to refine or expand coverage, and
       - One or more get_file calls to retrieve the full content of the most relevant documents.
   - For each search or get_file call, explain (in 2–3 sentences) why this action is necessary and how it helps answer the question.

3) Third iteration:
   - Analyze the retrieved document contents from get_file.
   - Synthesize the information from these documents into a final answer to the user.

IMPORTANT:
- At every step, explicitly explain your reasoning for each search query or file retrieval.
- Use only facts found in the documentation knowledge base.
- Do not introduce outside knowledge or assumptions.
- If the answer cannot be found in the retrieved documents, clearly inform the user.

Additional notes:
- The knowledge base is entirely about Evidently, so you do not need to include the word "evidently" in search queries.
- Prefer retrieving and analyzing full documents (via get_file) before producing the final answer.
"""

In [19]:
from toyaikit.tools import Tools

agent_tools = Tools()
agent_tools.add_tools(search_tools)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'get_file',
  'description': "Retrieve a file's contents by filename.\n\nArgs:\n    filename (str): The filename of the file to retrieve.\n\nReturns:\n    str: The file contents if found, otherwise an error message.",
  'parameters': {'type': 'object',
   'properties': {'filename': {'type': 'string',
     'description': 'filename parameter'}},
   'required': ['filename'],
   'additionalProperties': False}},
 {'type': 'function',
  'name': 'search',
  'description': 'Search the index for results matching a query and highlight them.\n\nArgs:\n    query (str): The search query to look up in the index.\n\nReturns:\n    List[Dict[str, Any]]: A list of highlighted search result objects.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [20]:
from openai import OpenAI
from toyaikit.llm import OpenAIClient

from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

In [21]:
llm_client = OpenAIClient(
    model="gpt-4o-mini",
    client=OpenAI()
)

chat_interface = IPythonChatInterface()
runner_callback = DisplayingRunnerCallback(chat_interface=chat_interface) 

In [22]:
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [23]:
result = agent.loop(
    query + ' show me the code',
    callback=runner_callback
)

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received
